In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
import os

import tensorflow_hub as hub
import tensorflow as tf

import torchaudio
import torch
from torch.utils.data import DataLoader, Dataset


df = pd.read_csv('/kaggle/input/birdclef-2023/train_metadata.csv')
AUDIO_PATH = Path('/kaggle/input/birdclef-2023/train_audio')
model = hub.load('https://kaggle.com/models/google/bird-vocalization-classifier/frameworks/TensorFlow2/variations/bird-vocalization-classifier/versions/2')
model_labels_df = pd.read_csv(hub.resolve('https://kaggle.com/models/google/bird-vocalization-classifier/frameworks/tensorFlow2/variations/bird-vocalization-classifier/versions/2') + "/assets/label.csv")

SAMPLE_RATE = 32000
WINDOW = 5*SAMPLE_RATE

In [ ]:
bc2023_labels = sorted(df.primary_label.unique())
label_to_index = {v: k for k, v in enumerate(bc2023_labels)}
model_labels = {v: k for k, v in enumerate(model_labels_df.ebird2021)}
model_bc2023_indexes = [model_labels[label] if label in model_labels else -1 for label in bc2023_labels]

# Save embeddings and predictions for every 5 sec non-overlapping audio

In [ ]:
# use a torch dataloader to decode audio in parallel on CPU while GPU is running
class AudioDataset(Dataset):
    def __len__(self):
        return len(df)
    def __getitem__(self, i):
        filename = df.filename[i]
        audio = torchaudio.load(AUDIO_PATH / filename)[0].numpy()[0]
        return audio, filename
dataloader = DataLoader(AudioDataset(), batch_size=1, num_workers=os.cpu_count())


# embeddings are formated like {"filename": np.array(nx1280)} 
# (where n = the number of non overlapping 5 sec chunks in the audio)
all_embeddings = {}

# predictiones formated like {"filename": np.array(nx264)} 
all_predictions = {}

with tf.device('/gpu:0'):
    for audio, filename in tqdm(dataloader):
        audio = audio[0]
        filename = filename[0]
        file_embeddings = []
        file_predictions = []
        for i in range(0, len(audio), WINDOW):
            clip = audio[i:i+WINDOW]
            if len(clip) < WINDOW:
                clip = np.concatenate([clip, np.zeros(WINDOW - len(clip))])
            result = model.infer_tf(clip[None, :])
            file_embeddings.append(result[1][0].numpy())
            prediction = np.concatenate([result[0].numpy(), -100], axis=None) # add -100 logit for unpredicted birds
            file_predictions.append(prediction[model_bc2023_indexes])
        all_embeddings[filename] = np.stack(file_embeddings)
        all_predictions[filename] = np.stack(file_predictions)

torch.save(all_embeddings, 'embeddings.pt')
torch.save(all_predictions, 'predictions.pt')

# Scores of predictions on the first 5 seconds of each recording

In [ ]:
predicted_classes = torch.tensor([row[0].argmax() for row in all_predictions.values()])
actual_classes = torch.tensor([label_to_index[label] for label in df.primary_label])
correct = predicted_classes == actual_classes
accuracy = correct.float().mean()
accuracy

In [ ]:
logits = torch.stack([torch.tensor(row[0]) for row in all_predictions.values()])
ce_loss = torch.nn.CrossEntropyLoss()(logits, actual_classes)
ce_loss

In [ ]:
actual_probs = torch.eye(len(bc2023_labels))[actual_classes]
bce_loss = torch.nn.BCEWithLogitsLoss()(logits, actual_probs)
bce_loss

In [ ]:
import pandas as pd
import sklearn.metrics

def padded_cmap(solution, submission, padding_factor=5):
    #solution = solution.drop(['row_id'], axis=1, errors='ignore')
    #submission = submission.drop(['row_id'], axis=1, errors='ignore')
    new_rows = []
    for i in range(padding_factor):
        new_rows.append([1 for i in range(len(solution.columns))])
    new_rows = pd.DataFrame(new_rows)
    new_rows.columns = solution.columns
    padded_solution = pd.concat([solution, new_rows]).reset_index(drop=True).copy()
    padded_submission = pd.concat([submission, new_rows]).reset_index(drop=True).copy()
    score = sklearn.metrics.average_precision_score(
        padded_solution.values,
        padded_submission.values,
        average='macro',
    )
    return score

In [ ]:
solution = pd.DataFrame(actual_probs.numpy(), columns=bc2023_labels)
padded_cmap(
    solution=solution,
    submission=pd.DataFrame(torch.softmax(logits, 1).numpy(), columns=bc2023_labels),
)

In [ ]:
padded_cmap(
    solution=solution,
    submission=pd.DataFrame(torch.sigmoid(logits).numpy(), columns=bc2023_labels),
)

# Analysis

In [ ]:
df['correct'] = correct.bool().numpy()

In [ ]:
import plotly.express as px

px.histogram(
    df,
    title="Distribution of accuracy",
    x='primary_label',
    color='correct',
).update_xaxes(categoryorder="total descending").show()

# Failure examples

In [ ]:
from IPython.display import Audio
import torchaudio
import matplotlib.pyplot as plt

compute_melspec = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_mels=128,
    n_fft=2048, 
    hop_length=512,
    f_min=0,
    f_max=SAMPLE_RATE // 2,
)

power_to_db = torchaudio.transforms.AmplitudeToDB(
    stype="power",
    top_db=80.0,
)

def show_bird(index, start=0):
    audio = torchaudio.load(AUDIO_PATH / df.filename[index], start, start+WINDOW)[0][0]
    display(df.iloc[index])
    display(Audio(audio, rate=SAMPLE_RATE))
    plt.figure(figsize=(12, 2.5))
    plt.subplot(121)
    plt.plot(audio)
    plt.gca().get_xaxis().set_visible(False)
    plt.subplot(122)
    plt.imshow(power_to_db(compute_melspec(audio)))
    plt.show()
    return 


# filter out birds that the model doesn't predict
missing_birds = set(np.array(bc2023_labels)[np.array(model_bc2023_indexes) == -1]) 
probs = torch.sigmoid(logits)

for count, i in enumerate(df[~df.correct & ~df.primary_label.isin(missing_birds)].sample(10).index):
    print('### EXAMPLE', count+1,'###')
    correct_class = df.primary_label[i]
    predicted_class_index = label_to_index[correct_class]
    predicted_prob_for_correct_label = probs[i, predicted_class_index]
    rank = 1 + sorted(probs[i], reverse=True).index(predicted_prob_for_correct_label)
    
    predicted_class_index = probs[i].argmax().item()
    predicted_clas = bc2023_labels[predicted_class_index]
    max_predicted_prob = probs[i][predicted_class_index]
    
    print(f'correct was {correct_class}, given prob {predicted_prob_for_correct_label} and ranked #{rank}')
    print(f'predicted {predicted_clas}, with prob {max_predicted_prob}')
    print()
    show_bird(i)